# Benchmark de EDs para Escalonamento e Priorização de Pacotes em Redes Restritas

## Table of contents

- Introdução
- Dados de teste
- EDs (Binary Heaps, Fibonacci Heaps, Pairing Heaps)
- Testes

## Introdução


In [2]:
%pip install pandas numpy gdown

  Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 705.7 kB/s eta 0:00:001m756.9 kB/s eta 0:00:01
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached charset_normalizer-3.4.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━

In [1]:
#@title Imports
import pandas as pd
import numpy as np
import gdown
import math

## Dados de teste

A primeira célula pega os arquivos direto da fonte original e trata do zero, a segunda célula pega os arquivos já tratados prontos para usar

No banco de dados MIMIC-III, a coluna **warning** na tabela chartevents atua como um sinalizador binário (0 ou 1) que indica se um alerta clínico ou uma observação manual foi documentado pelo profissional de saúde. [[1]](https://github.com/MIT-LCP/mimic-code/issues/472)

In [ ]:
#@title Obtendo dados de teste e salvando no drive

# Dados reais
# !wget -O chartevents.csv https://physionet.org/files/mimiciii-demo/1.4/CHARTEVENTS.csv?download
# chartevents = pd.read_csv("chartevents.csv")
# warning_stream = chartevents[["charttime", "warning"]].sort_values(by="charttime")
# warning_stream.to_csv("real_warning_stream.csv", index=False)

# Dados sintéticos
def sintetic_data(prob=0.20,n_linhas=1000,data_inicio='2026-01-01',data_fim='2026-12-31'):
  datas = pd.date_range(start=data_inicio, end=data_fim, freq='s')
  datas_registro = np.random.choice(datas, size=n_linhas, replace=True)
  datas_registro.sort()

  valores = np.random.choice([0, 1], size=n_linhas, p=[1-prob, prob])

  df_sintetico = pd.DataFrame({
      'charttime': datas_registro,
      'warning': valores
  })
  df_sintetico.to_csv(f'sintetic_prob_{int(prob*100)}.csv', index=False)
  return df_sintetico

sintetic_02 = sintetic_data(prob=0.02, n_linhas=10_000)
sintetic_20 = sintetic_data(n_linhas=10_000)
sintetic_50 = sintetic_data(prob=0.5, n_linhas=10_000)
sintetic_70 = sintetic_data(prob=0.7, n_linhas=10_000)

In [2]:
#@title Obtendo dados salvos no drive (já tratados)
def download_datasets():
  datasets = []
  file_ids = [("1J9g0S2pNts1Wc_ejdY2ASPIYlr38N2OX", "real_warning_stream.csv"),
              ("1Fbphw2X8ApaUnEdkhFQpsjBbP8i1ZOCa", "sintetic_prob_2.csv"),
              ("11E6JdyYdaSEjjOu0tHH3djqMZI81e7jC", "sintetic_prob_20.csv"),
              ("1CBq248GaofeRLtRyMKxSvSqCiQbnC2mU", "sintetic_prob_50.csv"),
              ("1MN6bboPC94Y6pekbSpJ8-EWfQaV_UMDe", "sintetic_prob_70.csv")]
  for file_id in file_ids:
    url = f'https://drive.google.com/uc?id={file_id[0]}'
    output = file_id[1]
    gdown.download(url, output, quiet=False)
    datasets.append(pd.read_csv(file_id[1]))
  return datasets

real_warning_stream, sintetic_prob_2, sintetic_prob_20,sintetic_prob_50, sintetic_prob_70 = download_datasets()

Downloading...
From: https://drive.google.com/uc?id=1J9g0S2pNts1Wc_ejdY2ASPIYlr38N2OX
To: /home/bradachi/Documentos/gitpath/masters-eda/real_warning_stream.csv
100%|██████████| 17.1M/17.1M [00:02<00:00, 8.10MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Fbphw2X8ApaUnEdkhFQpsjBbP8i1ZOCa
To: /home/bradachi/Documentos/gitpath/masters-eda/sintetic_prob_2.csv
100%|██████████| 220k/220k [00:00<00:00, 1.12MB/s]
Downloading...
From: https://drive.google.com/uc?id=11E6JdyYdaSEjjOu0tHH3djqMZI81e7jC
To: /home/bradachi/Documentos/gitpath/masters-eda/sintetic_prob_20.csv
100%|██████████| 220k/220k [00:00<00:00, 985kB/s]
Downloading...
From: https://drive.google.com/uc?id=1CBq248GaofeRLtRyMKxSvSqCiQbnC2mU
To: /home/bradachi/Documentos/gitpath/masters-eda/sintetic_prob_50.csv
100%|██████████| 220k/220k [00:00<00:00, 1.11MB/s]
Downloading...
From: https://drive.google.com/uc?id=1MN6bboPC94Y6pekbSpJ8-EWfQaV_UMDe
To: /home/bradachi/Documentos/gitpath/masters-eda/sintetic_prob_70.csv
100%|███

In [3]:
#@title Quantidade de alarmes cardíacos críticos por df
print("real_warning_stream")
display(real_warning_stream['warning'].value_counts())
print("sintetic_prob_2")
display(sintetic_prob_2['warning'].value_counts())
print("sintetic_prob_20")
display(sintetic_prob_20['warning'].value_counts())
print("sintetic_prob_50")
display(sintetic_prob_50['warning'].value_counts())
print("sintetic_prob_70")
display(sintetic_prob_70['warning'].value_counts())

real_warning_stream


warning
0.0    374893
1.0      7386
Name: count, dtype: int64

sintetic_prob_2


warning
0    9817
1     183
Name: count, dtype: int64

sintetic_prob_20


warning
0    7993
1    2007
Name: count, dtype: int64

sintetic_prob_50


warning
0    5052
1    4948
Name: count, dtype: int64

sintetic_prob_70


warning
1    7003
0    2997
Name: count, dtype: int64

## EDs (Binary Heaps, Fibonacci Heaps, Pairing Heaps)

In [54]:
from functools import total_ordering
#@title Binary Heap
# Usei como base a implementação do GeeksforGeeks
# disponível em: https://www.geeksforgeeks.org/dsa/binary-heap/

@total_ordering
class Key:
  def __init__(self, warning, charttime):
    self.warning = warning
    self.charttime = charttime

  def _criterios(self):
    return (self.warning, self.charttime)

  def __eq__(self, outro):
    return self._criterios() == outro._criterios()

  def __lt__(self, outro):
    return self._criterios() <= outro._criterios()
  
  def __str__(self):
    return f"({self.warning}, {self.charttime})"

class MinBinaryHeap:
  def __init__(self):
    self.arr = []

  # Get the index of the left child
  def left(self, i): return 2 * i + 1

  # Get the index of the right child
  def right(self, i): return 2 * i + 2

  # Get the index of the parent
  def parent(self, i): return (i - 1) // 2

  # Return the minimum element without removing it
  def get_min(self):
    return self.arr[0] if self.arr else None

  # Insert a new key into the heap
  def insert(self, k):
    self.arr.append(k)
    i = len(self.arr) - 1

    # Fix the min heap property by bubbling up
    # O próprio Python já compara tuplas e objetos elemento por elemento
    # (ordem lexicográfica)
    while i > 0 and self.arr[self.parent(i)] > self.arr[i]:
      p = self.parent(i)
      self.arr[i], self.arr[p] = self.arr[p], self.arr[i]
      i = p

  # Remove and return the root (minimum) element
  def extract_min(self):
    if len(self.arr) <= 0: return None
    if len(self.arr) == 1: return self.arr.pop()

    res = self.arr[0]
    # Replace root with the last element and heapify down
    self.arr[0] = self.arr.pop()
    self.min_heapify(0)
    return res

  # Recursive method to fix the heap property downwards
  def min_heapify(self, i):
    l, r, n = self.left(i), self.right(i), len(self.arr)
    smallest = i

    # Find the smallest among root, left child, and right child
    if l < n and self.arr[l] < self.arr[smallest]: smallest = l
    if r < n and self.arr[r] < self.arr[smallest]: smallest = r

    # If the root is not the smallest, swap and continue heapifying
    if smallest != i:
      self.arr[i], self.arr[smallest] = self.arr[smallest], self.arr[i]
      self.min_heapify(smallest)
  
  def __str__(self):
    res = '['
    for key in self.arr:
      res += ' ' + key.__str__() + ','
    res += ']'
    return res

In [ ]:
#@title Fibonacci Heap
# Usei como base a implementação do GeeksforGeeks
# disponível em: https://www.geeksforgeeks.org/dsa/fibonacci-heap-in-python/

class FibonacciHeapNode:
  def __init__(self, key):
      self.key = key
      self.degree = 0
      self.parent = None
      self.child = None
      self.mark = False
      self.left = self
      self.right = self

class FibonacciHeap:
    def __init__(self):
        self.min_node = None
        self.num_nodes = 0

    def is_empty(self):
        return self.min_node is None

    def insert(self, key):
        node = FibonacciHeapNode(key)
        if self.min_node is None:
            self.min_node = node
        else:
            node.left = self.min_node
            node.right = self.min_node.right
            self.min_node.right = node
            node.right.left = node
            if node.key < self.min_node.key:
                self.min_node = node
        self.num_nodes += 1
        return node

    def minimum(self):
      if self.min_node is None:
          return None
      return self.min_node.key

    def merge(self, other_heap):
      if self.min_node is None:
          self.min_node = other_heap.min_node
      elif other_heap.min_node is not None:
          self.min_node.right.left = other_heap.min_node.left
          other_heap.min_node.left.right = self.min_node.right
          self.min_node.right = other_heap.min_node
          other_heap.min_node.left = self.min_node
          if other_heap.min_node.key < self.min_node.key:
              self.min_node = other_heap.min_node
      self.num_nodes += other_heap.num_nodes

    def _remove_from_root_list(self, node):
      if node == node.right:
          self.min_node = None
      else:
          node.left.right = node.right
          node.right.left = node.left
          if node == self.min_node:
              self.min_node = node.right

    def _link(self, node1, node2):
      self._remove_from_root_list(node2)
      node2.left = node2.right = node2
      node2.parent = node1
      if node1.child is None:
          node1.child = node2
      else:
          node2.left = node1.child
          node2.right = node1.child.right
          node1.child.right = node2
          node2.right.left = node2
      node1.degree += 1
      node2.mark = False

    def _consolidate(self):
        if self.num_nodes <= 1:
            return
            
        import math
        max_degree = math.ceil(math.log(self.num_nodes, 2)) + 1
        degree_table = [None] * (max_degree + 1)

        root_nodes = []
        current = self.min_node
        if current is not None:
            while True:
                root_nodes.append(current)
                current = current.right
                if current == self.min_node:
                    break
        
        for current in root_nodes:
            
            if current.parent is not None:
                continue
                
            degree = current.degree
            while degree_table[degree] is not None:
                other = degree_table[degree]
                if current.key > other.key:
                    current, other = other, current
                self._link(current, other)
                degree_table[degree] = None
                degree += 1
            degree_table[degree] = current

        self.min_node = None
        for node in degree_table:
            if node is not None:
                if self.min_node is None:
                    self.min_node = node
                    node.left = node
                    node.right = node
                else:
                    node.right = self.min_node.right
                    node.left = self.min_node
                    self.min_node.right.left = node
                    self.min_node.right = node
                    if node.key < self.min_node.key:
                        self.min_node = node

    def extract_min(self):
      min_node = self.min_node
      if min_node is not None:
          if min_node.child is not None:
              children = [child for child in self._iterate_flat(min_node.child)]
              for child in children:
                  child.parent = None
                  self.min_node.left.right = child
                  child.left = self.min_node.left
                  child.right = self.min_node
                  self.min_node.left = child
                  if child.key < self.min_node.key:
                      self.min_node = child
          self._remove_from_root_list(min_node)
          if min_node == min_node.right:
              self.min_node = None
          else:
              self.min_node = min_node.right
              self._consolidate()
          self.num_nodes -= 1
      return min_node.key

    def _iterate_flat(self, head_node):
        current = head_node
        while True:
            yield current
            current = current.right
            if current == head_node:
                break

    def _iterate_all(self, head_node):
        current = head_node
        while True:
            yield current
            if current.child is not None:
                for n in self._iterate_all(current.child):
                    yield n
            current = current.right
            if current == head_node:
                break
    
    def __str__(self):
        if self.min_node is None:
            return "[ ]"
            
        chaves = [node.key for node in self._iterate_all(self.min_node)]
        chaves = sorted(chaves)
        chaves_str = ", ".join(map(str, chaves))
        
        return f"[ {chaves_str} ]"



In [6]:
#@title Pairing Heap
# Usei como base a implementação do GeeksforGeeks
# disponível em: https://www.geeksforgeeks.org/dsa/pairing-heap/

class PairingHeapNode:
    def __init__(self, key_=None, leftChild_=None, nextSibling_=None):
        self.key = key_
        self.leftChild = leftChild_
        self.nextSibling = nextSibling_

    # Adds a child and sibling to the node
    def addChild(self, node):
        if(self.leftChild == None):
            self.leftChild = node
        else:
            node.nextSibling = self.leftChild
            self.leftChild = node

def Empty(node):
    return (node == None)

def Merge(A, B):

    # If any of the two-nodes is None
    # the return the not None node
    if(A == None):
        return B
    if(B == None):
        return A

    # To maintain the min heap condition compare
    # the nodes and node with minimum value become
    # parent of the other node
    if(A.key < B.key):
        A.addChild(B)
        return A
    B.addChild(A)
    return B

# Returns the root value of the heap

def Top(node):
    return node.key

# Function to insert the new node in the heap
def Insert(node, key):
    return Merge(node, PairingHeapNode(key,))

# This method is used when we want to delete root node
def TwoPassMerge(node):
    if(node == None or node.nextSibling == None):
        return node
    A = node
    B = node.nextSibling
    newNode = node.nextSibling.nextSibling

    A.nextSibling = None
    B.nextSibling = None

    return Merge(Merge(A, B), TwoPassMerge(newNode))

# Function to delete the root node in heap
def Delete(node):
    return TwoPassMerge(node.leftChild)

class PairingHeap:
    def __init__(self):
        self.root = None

    def Empty(self):
        return Empty(self.root)

    def Top(self):
        return Top(self.root)

    def Insert(self, key):
        self.root = Insert(self.root, key)

    def Delete(self):
        self.root = Delete(self.root)

    def Join(self, other):
        self.root = Merge(self.root, other.root)



# Testes

In [95]:
#@title Toy Test

def toy_test(ED):
    test_data = {'warning': [0,-1,0,0,-1,-1,-1,-1,0,0],
                'charttime': ['2102-08-31 17:00:00', '2102-08-31 17:04:00', 
                            '2102-08-31 17:04:00', '2102-08-31 17:04:00', 
                            '2102-08-31 17:04:00', '2102-08-31 17:07:00', 
                            '2102-08-31 17:07:00', '2102-08-31 17:07:00', 
                            '2102-08-31 17:07:00', '2102-08-31 17:08:00']}

    test_df = pd.DataFrame(test_data)
    data_structure = ED()

    for _, linha in test_df.iterrows():
        key = Key(warning=linha['warning'], charttime=linha['charttime'])
        data_structure.insert(key)
        print(data_structure)
    
    for _ in range(5):
        print(f"extraindo mínimo: {data_structure.extract_min()}") 
        print(data_structure)

        

In [98]:
toy_test(MinBinaryHeap)

[ (0, 2102-08-31 17:00:00),]
[ (-1, 2102-08-31 17:04:00), (0, 2102-08-31 17:00:00),]
[ (-1, 2102-08-31 17:04:00), (0, 2102-08-31 17:00:00), (0, 2102-08-31 17:04:00),]
[ (-1, 2102-08-31 17:04:00), (0, 2102-08-31 17:00:00), (0, 2102-08-31 17:04:00), (0, 2102-08-31 17:04:00),]
[ (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:04:00), (0, 2102-08-31 17:04:00), (0, 2102-08-31 17:04:00), (0, 2102-08-31 17:00:00),]
[ (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:07:00), (0, 2102-08-31 17:04:00), (0, 2102-08-31 17:00:00), (0, 2102-08-31 17:04:00),]
[ (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:07:00), (0, 2102-08-31 17:04:00), (0, 2102-08-31 17:00:00), (0, 2102-08-31 17:04:00), (-1, 2102-08-31 17:07:00),]
[ (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:07:00), (-1, 2102-08-31 17:07:00), (0, 2102-08-31 17:00:00), (0, 2102-08-31 17:04:00), (-1, 2102-08-31 17:07:00), (0, 2102-08-31 17:04:00),]
[ (-1, 2102-08-31 17:04

In [97]:
toy_test(FibonacciHeap)

[ (0, 2102-08-31 17:00:00) ]
[ (-1, 2102-08-31 17:04:00), (0, 2102-08-31 17:00:00) ]
[ (-1, 2102-08-31 17:04:00), (0, 2102-08-31 17:00:00), (0, 2102-08-31 17:04:00) ]
[ (-1, 2102-08-31 17:04:00), (0, 2102-08-31 17:00:00), (0, 2102-08-31 17:04:00), (0, 2102-08-31 17:04:00) ]
[ (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:04:00), (0, 2102-08-31 17:00:00), (0, 2102-08-31 17:04:00), (0, 2102-08-31 17:04:00) ]
[ (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:07:00), (0, 2102-08-31 17:00:00), (0, 2102-08-31 17:04:00), (0, 2102-08-31 17:04:00) ]
[ (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:07:00), (-1, 2102-08-31 17:07:00), (0, 2102-08-31 17:00:00), (0, 2102-08-31 17:04:00), (0, 2102-08-31 17:04:00) ]
[ (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:04:00), (-1, 2102-08-31 17:07:00), (-1, 2102-08-31 17:07:00), (-1, 2102-08-31 17:07:00), (0, 2102-08-31 17:00:00), (0, 2102-08-31 17:04:00), (0, 2102-08-31 17:04:00) ]
[ (-1, 2102-08-31 17:04

In [ ]:
# simulação de um aparelho coletando dados e processando

def simulation(ED, df):
  data_structure = ED()

  # insere os primeiros 1000
  df_1000 = df[:1000]
  for indice, linha in df_1000.iterrows():
    key = Key(warning=linha['warning'], charttime=linha['charttime'])
    data_structure.insert(key)


# for len(dataset)
  # envia 1 (extract_min)
  # insere 1 (insert)